In [6]:
%pip install pycayennelpp requests
import requests
import json
from cayennelpp import LppFrame
import base64
#import pandas
from IPython.display import display, Markdown, Latex
display.max_rows = 4000
display.max_seq_items = 4000
DISPLAY_DOCKLOGS = True

test = "✅""❌"


[notice] A new release of pip available: 22.2.1 -> 24.0
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
iot_lora_service = "http://localhost:6041"
url = f"{iot_lora_service}/iot/services"
url_devices = f"{iot_lora_service}/iot/devices"
did = "104868453"
offset = "1"
device_id = f"hsensor{did}_{offset}"


In [4]:
mosquitto_command_template = """
```
mosquitto_pub  -u admin -P password -t v3/sogeiTTN/devices/{deviceid}/up -m '{{
    "app_id": "sogeiTTN",
    "dev_id": "{deviceid}",
    "hardware_serial": "0102030405060708",
    "port": 1,
    "counter": 2,
    "is_retry": false,
    "confirmed": false,
    "payload_raw": "{payload}"
}}'
```
"""

In [84]:
#url = "https://6041-aquarta-bigdataprojecta-oe6lq79ertz.ws-eu114.gitpod.io/iot/services"
#iot_lora_service = "http://iotagent-lora:6041"

service_dict = {
  "services": [
    {
      "entity_type": "HeightSensor",
      "apikey": "",
      "resource": "70B3D57ED00006F2",
      "cbroker": "http://orion:1026",
      #"type": "Device",

      "static_attributes": [
        {
          "name": "category",
          "type": "Property",
          "value": "sensor"
        },
        {
          "name": "supportedProtocol",
          "type": "Property",
          "value": "ul20"
        },
        {
            "name": "controlledAsset",
            "type": "Relationship"
        }
      ],
      "internal_attributes": {
        "lorawan": {
          "application_server": {
            "host": "mqtt",
            "username": "admin",
            "password": "password",
            "provider": "TTN"
          },
          "app_eui": "70B3D57ED00006F2",
          "application_id": "sogeiTTN",
          "application_key": "BE6996EEE2B2D6AFFD951383C1F3C3BD",
          "data_model": "cayennelpp"
        }
      }
    }
  ]
}

service_dict["services"][0].update({
  "attributes": [
        {
            "object_id": "gps_1",
            "name": "location",
            "type": "Property"
        }
      ]
    }
)
payload = json.dumps(service_dict)
headers = {
  'fiware-service': 'openiot',
  'fiware-servicepath': '/',
  'Content-Type': 'application/json'
}
#print(payload)
response = requests.request("POST", url, headers=headers, data=payload)
print(response.status_code)
print(response.text)

201
{}


In [8]:
# Add device
payload = json.dumps({
  "devices": [
    {
      "device_id": device_id,
      "entity_name": f"urn:ngsi-ld:Device:hsensor:{device_id}",
      "entity_type": "HeightSensor",
      "internal_attributes": {
       "lorawan": {
            "application_server": {
                "host": "mqtt",
                "username": "admin",
                "password": "password",
                "provider": "TTN"
            },
            "app_eui": "70B3D57ED00006F2",
            "application_id": "sogeiTTN",
            "application_key": "BE6996EEE2B2D6AFFD951383C1F3C3BD",
            "data_model": "cayennelpp"
        }
      },
       "attributes": [
        {
          "object_id": "gps_1",
          "name": "location",
          "type": "Property"
        }
      ],
      "static_attributes": [
        {
          "name": "controlledAsset",
          "type": "Relationship",
          "object": f"urn:ngsi-ld:Building:waybridge{did}"
        }
      ]
    }
  ]
})
headers = {
  'fiware-service': 'openiot',
  'fiware-servicepath': '/',
  'Content-Type': 'application/json'
}
print(payload)
response = requests.request("POST", url_devices, headers=headers, data=payload)
print(response.status_code)
print(response.text)


orion_url = "http://localhost:1026/ngsi-ld/v1/subscriptions/"
payload = json.dumps({
  "description": "Notify flask of all height changes",
  "name": "Notify_flask_sens_motion",
  "type": "Subscription",
  "entities": [
    {
      "type": "HeightSensor"
    }
  ],
  "notification": {
    "format": "normalized",
    "endpoint": {
      "uri": "http://flaskdash:8000/api/v1/sens_notify",
      "accept": "application/json"
    },
    "showChanges": True
  },
  "@context": "http://context/datamodels.context.jsonld"
})
headers = {
  'Fiware-Service': 'openiot',
  'Fiware-ServicePath': '/',
  'NGSILD-Tenant': 'openiot',
  'Content-Type': 'application/ld+json'
}

# response = requests.request("POST", orion_url, headers=headers, data=payload)

# print(response.text)



{"devices": [{"device_id": "hsensor104868453_1", "entity_name": "urn:ngsi-ld:Device:hsensor:hsensor104868453_1", "entity_type": "HeightSensor", "internal_attributes": {"lorawan": {"application_server": {"host": "mqtt", "username": "admin", "password": "password", "provider": "TTN"}, "app_eui": "70B3D57ED00006F2", "application_id": "sogeiTTN", "application_key": "BE6996EEE2B2D6AFFD951383C1F3C3BD", "data_model": "cayennelpp"}}, "attributes": [{"object_id": "gps_1", "name": "location", "type": "Property"}], "static_attributes": [{"name": "controlledAsset", "type": "Relationship", "object": "urn:ngsi-ld:Building:waybridge104868453"}]}]}
201
{}


In [9]:
frame = LppFrame()
location = (13.1,13.1,13.1)
frame.add_location(1, *location)

# get byte buffer in CayenneLPP format
buffer = bytes(frame)

gfg = base64.b64encode(buffer) 

mosquitto_command = mosquitto_command_template.format(payload=gfg.decode(),deviceid=device_id)
display(Markdown(f'✅  **location {location}** `{mosquitto_command}` '))
!{mosquitto_command}
iota_out = !docker logs  -t -n 70 fiware-iota-lora
iota_out = "\n".join(iota_out)
if DISPLAY_DOCKLOGS :
    print(iota_out)
if iota_out.find("Observations sent to CB successfully")>0:
    display(Markdown(f'✅  Sent  '))
else:
    display(Markdown(f'❌  FAIL ❌ '))


✅  **location (13.1, 13.1, 13.1)** `
```
mosquitto_pub  -u admin -P password -t v3/sogeiTTN/devices/hsensor104868453_1/up -m '{
    "app_id": "sogeiTTN",
    "dev_id": "hsensor104868453_1",
    "hardware_serial": "0102030405060708",
    "port": 1,
    "counter": 2,
    "is_retry": false,
    "confirmed": false,
    "payload_raw": "AYgB/7gB/7gABR4="
}'
```
` 

2024-06-15T18:55:49.021299040Z     "json": [
2024-06-15T18:55:49.021303920Z         {
2024-06-15T18:55:49.021308220Z             "@context": "http://context/datamodels.context.jsonld",
2024-06-15T18:55:49.021313010Z             "location": {
2024-06-15T18:55:49.021317040Z                 "type": "Property",
2024-06-15T18:55:49.021321440Z                 "value": {
2024-06-15T18:55:49.021325440Z                     "latitude": 13.1,
2024-06-15T18:55:49.021329390Z                     "longitude": 13.1,
2024-06-15T18:55:49.021333380Z                     "altitude": 13.1
2024-06-15T18:55:49.021337720Z                 },
2024-06-15T18:55:49.021341900Z                 "observedAt": "2024-06-15T18:55:49.017Z"
2024-06-15T18:55:49.021346230Z             },
2024-06-15T18:55:49.021350260Z             "category": {
2024-06-15T18:55:49.021354460Z                 "type": "Property",
2024-06-15T18:55:49.021358990Z                 "value": "sensor",
2024-06-15T18:55:49.021363240Z                 "obse

✅  Sent  

In [10]:
frame = LppFrame()
location = (13.1,13.1,16.1)
frame.add_location(1, *location)

# get byte buffer in CayenneLPP format
buffer = bytes(frame)

gfg = base64.b64encode(buffer) 

mosquitto_command = mosquitto_command_template.format(payload=gfg.decode(),deviceid=device_id)
display(Markdown(f'✅  **location {location}** `{mosquitto_command}` '))
!{mosquitto_command}
iota_out = !docker logs  -t -n 75 fiware-iota-lora
iota_out = "\n".join(iota_out)
if DISPLAY_DOCKLOGS :
    print(iota_out)
if iota_out.find("Observations sent to CB successfully")>0:
    display(Markdown(f'✅  Sent  '))
else:
    display(Markdown(f'❌  FAIL ❌ '))

✅  **location (13.1, 13.1, 16.1)** `
```
mosquitto_pub  -u admin -P password -t v3/sogeiTTN/devices/hsensor104868453_1/up -m '{
    "app_id": "sogeiTTN",
    "dev_id": "hsensor104868453_1",
    "hardware_serial": "0102030405060708",
    "port": 1,
    "counter": 2,
    "is_retry": false,
    "confirmed": false,
    "payload_raw": "AYgB/7gB/7gABko="
}'
```
` 

2024-06-15T18:56:35.243093337Z     },
2024-06-15T18:56:35.243096067Z     "json": [
2024-06-15T18:56:35.243098847Z         {
2024-06-15T18:56:35.243102817Z             "@context": "http://context/datamodels.context.jsonld",
2024-06-15T18:56:35.243107707Z             "location": {
2024-06-15T18:56:35.243110727Z                 "type": "Property",
2024-06-15T18:56:35.243113527Z                 "value": {
2024-06-15T18:56:35.243116247Z                     "latitude": 13.1,
2024-06-15T18:56:35.243119017Z                     "longitude": 13.1,
2024-06-15T18:56:35.243121807Z                     "altitude": 16.1
2024-06-15T18:56:35.243124497Z                 },
2024-06-15T18:56:35.243127187Z                 "observedAt": "2024-06-15T18:56:35.242Z"
2024-06-15T18:56:35.243130037Z             },
2024-06-15T18:56:35.243132687Z             "category": {
2024-06-15T18:56:35.243135527Z                 "type": "Property",
2024-06-15T18:56:35.243138277Z                 "value": "sensor",
2024-06-15T18:

✅  Sent  

In [108]:
frame = LppFrame()
location = (13.1,13.1,19.1)
frame.add_location(1, *location)

# get byte buffer in CayenneLPP format
buffer = bytes(frame)

gfg = base64.b64encode(buffer) 

mosquitto_command = mosquitto_command_template.format(payload=gfg.decode(),deviceid=device_id)
display(Markdown(f'✅  **location {location}** `{mosquitto_command}` '))
!{mosquitto_command}
iota_out = !docker logs  -t -n 75 fiware-iota-lora
iota_out = "\n".join(iota_out)
if DISPLAY_DOCKLOGS :
    print(iota_out)
if iota_out.find("Observations sent to CB successfully")>0:
    display(Markdown(f'✅  Sent  '))
else:
    display(Markdown(f'❌  FAIL ❌ '))

✅  **location (13.1, 13.1, 19.1)** `
```
mosquitto_pub  -u admin -P password -t v3/sogeiTTN/devices/hsensor297484554_1/up -m '{
    "app_id": "sogeiTTN",
    "dev_id": "hsensor297484554_1",
    "hardware_serial": "0102030405060708",
    "port": 1,
    "counter": 2,
    "is_retry": false,
    "confirmed": false,
    "payload_raw": "AYgB/7gB/7gAB3Y="
}'
```
` 

✅  Sent  

In [83]:


url = f"{iot_lora_service}/iot/services/?resource=70B3D57ED00006F2&apikey="

payload = {}
headers = {
  'fiware-service': 'openiot',
  'fiware-servicepath': '/'
}

response = requests.request("DELETE", url, headers=headers, data=payload)
print(response.status_code)
print(response.text)

404
{"name":"DEVICE_GROUP_NOT_FOUND","message":"Couldn\t find device group for fields: [\"resource\",\"apikey\"] and values: {\"resource\":\"70B3D57ED00006F2\"}"}
